# Installiere ultralytics Bibliothek

In [ ]:
!pip install ultralytics

## Import Libraries

In [ ]:
import os
from zipfile import ZipFile
from ultralytics import YOLO

In [ ]:
path = "D:/Studium/5. Semester/PA2/Glass-Defect-Detection-Evaluating-Object-Detection-Models/Coding/Model_Code/YOLO"
print(path)

## Load in the data and unzip it

In [ ]:
# Create the directory if it doesn't exist
extract_dir = "data"
os.makedirs(extract_dir, exist_ok=True)

with ZipFile("Kaggle_Dataset.v1i.yolov11.zip", "r") as data_set:
    data_set.extractall(extract_dir)

## Load the Model


In [ ]:
model = YOLO("yolo11n.pt")

## Create configurations


In [ ]:
DATA_YAML = path +"/data/data.yaml"
MODEL_PATH = "yolo11n.pt"
data_set_dir = "data"

## Fine-Tuning with Hyperparameters

In [ ]:
results = model.train(
    data = DATA_YAML,
    epochs = 100,
    batch = 16,
    lr0 = 0.001,
    val = True
)

#### Testing weigths on the test dataset

In [ ]:
model = YOLO(path + "/runs/detect/train2/weights/best.pt")

In [ ]:
metrics = model.val(
    data = DATA_YAML,
    split  = "test",
    batch = 16
    )

print(metrics.box.map)

In [ ]:
metrics.results_dict

In [ ]:
??metrics

In [ ]:
import time
import cv2
import torch # Nur zur Überprüfung, ob eine GPU verfügbar ist

# Laden des besten Modells
model = YOLO('best.pt')

# Wichtig: Gerät auf CPU setzen! (Auch wenn eine GPU vorhanden ist)
device = 'cpu'
if torch.cuda.is_available():
    print("GPU erkannt, aber Inferenz wird auf CPU forciert.")

# Testbild laden (oder eine Schleife über den gesamten Test-Datensatz)
image = cv2.imread(path + '/test/images/image_10_jpg.rf.86f3dc2cbb9d15047ab07ff149cf3a2f.jpg')

# Inferenz-Schleife für genaue Messung
num_runs = 100
start_time = time.time()

for _ in range(num_runs):
    # Inferenz mit YOLOv11 Nano auf der CPU
    results = model(
        image,
        device=device,
        verbose=False # Ausgabe unterdrücken
    )
    # Beachten Sie, dass das Post-Processing (NMS) in der Zeit enthalten ist.

end_time = time.time()
avg_time_ms = ((end_time - start_time) / num_runs) * 1000
fps = 1000 / avg_time_ms

print(f"\nDurchschnittliche Inferenzzeit auf {device}: {avg_time_ms:.2f} ms")
print(f"Geschwindigkeit: {fps:.2f} FPS")

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Matrix mit der Anordnung aus dem BILD (Predicted x True)
cm_raw_data = np.array([
    [27, 1],
    [0, 0] 
])

# Standard-Labels (True=Reihe, Predicted=Spalte)
standard_labels = ['DEFECT', 'no defect']

# Transponieren der Matrix: (Predicted x True) --> (True x Predicted)
# Dies korrigiert die vertauschte Achsenanordnung des Bildes.
cm_final_data = cm_raw_data.T

# --- 2. Plot der korrigierten 2x2 Matrix ---
plt.figure(figsize=(7, 6))
sns.heatmap(
    cm_final_data, 
    annot=True, 
    fmt='d', 
    cmap='Blues', 
    cbar=True,
    xticklabels=standard_labels,
    yticklabels=standard_labels
)

plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix YOLO, Kaggle')
plt.show()

# Print der finalen Matrix
# print("\nKonsolidierte 2x2 Matrix (True=Reihe vs. Predicted=Spalte):\n")
# df = pd.DataFrame(cm_final_data, index=standard_labels, columns=standard_labels)
# print(df)